***IMPORT DE LIBRERIAS A UTILIZAR***

In [28]:
import requests
from bs4 import BeautifulSoup
import csv
import pandas as pd
import time 
import psycopg2

***CONEXION CON BOOKS TO SCRAPE***

In [8]:
try:
    #indicamos la url a la que realizaremos peticiones GET
    book_scrape = "https://books.toscrape.com/"
    #donde guardaremos la respuesta que obtenemos de la pagina WEB
    response = requests.get(book_scrape)
    #creamos un objeto soup para parsearlo con el contenido html 
    soup = BeautifulSoup(response.text, "html.parser")
    #verificar estado de conexion 200 = exitosa
    if response.status_code == 200:
        print("Conexion exitosa...")   
except (Exception,KeyboardInterrupt) as e:
    print(f"Hubo un error {e}")


Conexion exitosa...


***ITERACION POR PAGINAS PARA SCRAPEO DE INFORMACION DE LIBROS***

In [ ]:
libro_datos = [] #creamos una lista que almacenara los datos de los libros
for page_num in range(1,51):
    #realizara las iteraciones del for dentro de la url de cada pagina del catalogo
    books_pages = f'https://books.toscrape.com/catalogue/page-{page_num}.html'
    response = requests.get(books_pages)
    soup = BeautifulSoup(response.content, 'html.parser') # obtener el html 

    libros = soup.find_all('h3')

    #iterar sobre la lista de titulos de libros
    for libro in libros:
        try:
            #encontrar la url vinculada al libro (libro.find)
            libro_url = libro.find('a')['href']
            #accede a la url de books to scrape + a la url de el libro
            libro_response = requests.get('https://books.toscrape.com/catalogue/' + libro_url)
            #obtenemos el html de la pagina de la url del libro 
            libro_soup = BeautifulSoup(libro_response.content, "html.parser")  
            
            #busqueda de datos de los libros mediante etiquetas HTML
            #encontrar el titulo del libro mediante la etiqueta del h1
            titulo = libro_soup.find('h1').text 
            #accede a la etiqueta ul donde encuentra la ruta de busqueda donde se encuentra la categoria del libro
            categoria = libro_soup.find('ul', class_ ="breadcrumb").find_all('a')[2].text.strip()
            calificacion = libro_soup.find('p', class_ ='star-rating')['class'][1]
            precio = libro_soup.find('p', class_ ="price_color").text.strip()

            libro_datos.append([titulo,categoria,calificacion,precio])
        except (Exception,KeyboardInterrupt) as e:
            print(f"Hubo un error {e}")
            continue

Hubo un error 


***CONVERSION DE LISTA A CSV MEDIANTE PANDAS***

In [7]:

#Crear el DataFrame
df = pd.DataFrame(libro_datos, columns=["titulo", "categoria", "calificacion", "precio"])

#Limpiar precio
df["precio"] = df["precio"].str.replace("£", "").astype(float)

#Convertir calificacion a número
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
df["calificacion"] = df["calificacion"].map(rating_map)

#Guardar CSV
df.to_csv("../data/libros_scrapeados.csv", index=False)
print(df.head())

NameError: name 'libro_datos' is not defined

***OBTENCION DE KEY DE AUTORES MEDIANTE TITULO DE OBRA (OPEN-LIBRARY)***

In [ ]:
columna_titulo= ['titulo']
#leemos el csv para poder realizar acciones sobre los campos del csv
datos = pd.read_csv("../data/libros_scrapeados.csv")
titulos = datos['titulo'].tolist()
keys = []
for titulo in titulos:

    try:
        titulo_corto = titulo.split(":")[0].split("(")[0].split(",")[0].strip()
        params = {"title": titulo_corto, "limit": 1}
        url_opl = "https://openlibrary.org/search.json"
        response = requests.get(url_opl, params=params)
        data = response.json()

        doc= data["docs"][0]

        opl_id = doc["author_key"][0]

        keys.append([opl_id])
    except Exception as e:
        print(f"No encontrado: {titulo} - {e}")
        keys.append([None])
        continue

    print (keys)

***CONVERSION DE LISTA DE AUTOR_KEYS A CSV***

In [ ]:
#convertir una lista en un dataframe
df = pd.DataFrame(keys, columns=["autor_key"])
# convertir dataframe a csv
df.to_csv("autor_key.csv", index=False)
df

,autor_key
0,OL548174A
1,OL39232A
2,OL300477A
3,OL1433006A
4,OL3778242A
...,...
995,OL22098A
996,OL7423510A
997,OL7032059A
998,OL22258A


***ENRIQUECIMIENTO DE DATOS DE AUTOR MEDIANTE AUTOR_KEY (OPEN-LIBRARY)***

In [ ]:
columna_autor_key= ['autor_key']
#leemos el csv para poder realizar acciones sobre los campos del csv
dato_key = pd.read_csv("../data/autor_key.csv")
autor_keys = dato_key['autor_key'].tolist()

autor_datos = []

for key in autor_keys:
    if pd.isna(key):
        autor_datos.append([None, None, None, None])
        continue 
    try:
        url_autor = f"https://openlibrary.org/authors/{key}.json"
        url_works = f"https://openlibrary.org/authors/{key}/works.json"

        response_autor = requests.get(url_autor)
        response_works = requests.get(url_works)

        data_works = response_works.json()
        data_autor = response_autor.json()

        birth_date = data_autor.get("birth_date", None)
        nombre = data_autor.get("personal_name",None)
        fecha_creacion = data_autor.get("created", {}).get("value", None)

        total_obras = data_works.get("size", None)

        anho_nacimiento = None
        if birth_date:
            for i in range(len(birth_date) - 3):
                parte = birth_date[i:i+4]
                if parte.isdigit():
                    anho_nacimiento = int(parte)
                    break
        autor_datos.append([anho_nacimiento,nombre,fecha_creacion,total_obras])
        time.sleep(0.5)  # al final de cada iteración del loop
    except Exception as e:
        print(f"Error: {key} - {e}")
        autor_datos.append([None, None, None, None])
        continue
    
    print(f"Nombre: {nombre}")
    print(f"Anho: {anho_nacimiento}")
    print(f"Creacion: {fecha_creacion}")
    print(f"Cantidad Obras: {total_obras}")
    print("--------------")

Nombre: Shel Silverstein
Anho: None
Creacion: 2008-04-01T03:28:50.625462
Cantidad Obras: 152
--------------
Nombre: Sarah Waters
Anho: 1966
Creacion: 2008-04-01T03:28:50.625462
Cantidad Obras: 36
--------------
Nombre: Michel Houellebecq
Anho: 1956
Creacion: 2008-04-01T03:28:50.625462
Cantidad Obras: 85
--------------
Nombre: Gillian Flynn
Anho: 1971
Creacion: 2008-04-01T03:28:50.625462
Cantidad Obras: 43
--------------
Nombre: Yuval N. Harari
Anho: 1976
Creacion: 2008-04-30T20:50:18.033121
Cantidad Obras: 53
--------------
Nombre: None
Anho: None
Creacion: None
Cantidad Obras: 8
--------------
Nombre: Don Raskin
Anho: None
Creacion: 2019-07-19T11:37:45.718721
Cantidad Obras: 1
--------------
Nombre: None
Anho: None
Creacion: 2025-11-27T11:54:06.294424
Cantidad Obras: 1
--------------
Nombre: Brown, Daniel
Anho: 1951
Creacion: 2008-04-01T03:28:50.625462
Cantidad Obras: 6
--------------
Nombre: Aracelis Girmay
Anho: None
Creacion: 2008-04-01T03:28:50.625462
Cantidad Obras: 13
----------

***CONVERSION LISTA DE DATOS DE AUTOR A CSV MEDIANTE PANDAS***

In [4]:
#convertir una lista en un dataframe
df = pd.DataFrame(autor_datos, columns=["anho_nacimiento","nombre","fecha_creacion","total_obras"])
# convertir dataframe a csv
df.to_csv("autor_datos.csv", index=False)
df

,anho_nacimiento,nombre,fecha_creacion,total_obras
0,NaN,Shel Silverstein,2008-04-01T03:28:50.625462,152.0
1,1966.0,Sarah Waters,2008-04-01T03:28:50.625462,36.0
2,1956.0,Michel Houellebecq,2008-04-01T03:28:50.625462,85.0
3,1971.0,Gillian Flynn,2008-04-01T03:28:50.625462,43.0
4,1976.0,Yuval N. Harari,2008-04-30T20:50:18.033121,53.0
...,...,...,...,...
995,1832.0,"Carroll, Lewis",2008-04-01T03:28:50.625462,2637.0
996,NaN,NaN,2018-10-26T12:14:46.657904,53.0
997,NaN,Melanie Dickerson,2011-11-04T09:12:03.932079,32.0
998,1947.0,"Patterson, James",2008-04-01T03:28:50.625462,706.0


***ENRIQUECIMIENTO DE NACIONALIDAD DE AUTORES MEDIANTE NOMBRE DE AUTOR (WIKIPEDIA)***

In [9]:
columna_nombre= ['nombre']
#leemos el csv para poder realizar acciones sobre los campos del csv
dato_nombre = pd.read_csv("../data/autor_datos.csv")
autor_nombre = dato_nombre['nombre'].tolist()

nacionalidades = []
for nombre in autor_nombre:
    if pd.isna(nombre):
        nacionalidades.append(None)
        continue
    try:
        headers = {
            "User-Agent": "books-scrape-challenge/1.0 (hugo@email.com)"
        }
        nombre_limpio = nombre.strip().rstrip(".")
        url_wiki = f"https://en.wikipedia.org/api/rest_v1/page/summary/{nombre_limpio.replace(' ', '_')}"
        response_wiki = requests.get(url_wiki, headers=headers, timeout=10)

        data_wiki = response_wiki.json()
        description = data_wiki.get("description", None)
        
        if description and description != "Topics referred to by the same term":
            nacionalidad = description.split()[0]
        else:
            nacionalidad = None
        nacionalidades.append(nacionalidad)
        time.sleep(0.3)
        print(nacionalidades)
    except Exception as e:
        print(f"Error: {nombre} - {e}")
        nacionalidades.append(None)
        continue

['American']
['American', 'Welsh']
['American', 'Welsh', 'French']
['American', 'Welsh', 'French', 'American']
['American', 'Welsh', 'French', 'American', 'Israeli']
['American', 'Welsh', 'French', 'American', 'Israeli', None, None]
['American', 'Welsh', 'French', 'American', 'Israeli', None, None, None, None]
['American', 'Welsh', 'French', 'American', 'Israeli', None, None, None, None, 'American']
['American', 'Welsh', 'French', 'American', 'Israeli', None, None, None, None, 'American', None, 'English']
['American', 'Welsh', 'French', 'American', 'Israeli', None, None, None, None, 'American', None, 'English', 'American']
['American', 'Welsh', 'French', 'American', 'Israeli', None, None, None, None, 'American', None, 'English', 'American', 'Canadian']
['American', 'Welsh', 'French', 'American', 'Israeli', None, None, None, None, 'American', None, 'English', 'American', 'Canadian', 'English']
['American', 'Welsh', 'French', 'American', 'Israeli', None, None, None, None, 'American', Non

***CONVERSION LISTA DE NACIONALIDADES A CSV MEDIANTE PANDAS***

In [10]:
#convertir una lista en un dataframe
df = pd.DataFrame(nacionalidades, columns=["nacionalidad"])
# convertir dataframe a csv
df.to_csv("autor_nacionalidad.csv", index=False)
df 

,nacionalidad
0,American
1,Welsh
2,French
3,American
4,Israeli
...,...
995,British
996,NaN
997,NaN
998,American


***VERIFICAMOS CANTIDAD DE INFORMACION ALMACENADA EN LOS CSV***

In [25]:
#leer los csv a traves de la ruta de archivos
df_libros  = pd.read_csv("../data/libros_scrapeados.csv")
df_keys    = pd.read_csv("../data/autor_key.csv")
df_autores = pd.read_csv("../data/autor_datos.csv")
df_nacion  = pd.read_csv("../data/autor_nacionalidad.csv")
#imprimimos la cantidad de filas de cada csv para ver cantidad de informacion 
print(len(df_libros), len(df_keys), len(df_autores), len(df_nacion))

1000 1000 1000 1000


***CREACION DE TABLAS STAGING***

In [34]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    user="postgres",
    password="12345",
    database="ConsultaMortal"
)
cursor = conn.cursor()

staging_ddl = """
DROP TABLE IF EXISTS staging_books, staging_authors, staging_keys, staging_nationalities CASCADE;

CREATE TABLE staging_books (
    titulo       TEXT,
    categoria    TEXT,
    calificacion TEXT,
    precio       TEXT
);

CREATE TABLE staging_authors (
    anho_nacimiento TEXT,
    nombre          TEXT,
    fecha_creacion  TEXT,
    total_obras     TEXT
);

CREATE TABLE staging_keys (
    autor_key TEXT
);

CREATE TABLE staging_nationalities (
    nacionalidad TEXT
);
"""

cursor.execute(staging_ddl)
conn.commit()
print("Tablas staging creadas")

Tablas staging creadas


***CARGA DE DATOS A TABLAS STAGING***

In [35]:
# Cargar libros
with open("../data/libros_scrapeados.csv", "r", encoding="utf-8") as f:
    cursor.copy_expert("""
        COPY staging_books (titulo, categoria, calificacion, precio)
        FROM STDIN WITH (FORMAT csv, HEADER true, DELIMITER ',')
    """, f)

# Cargar autores
with open("../data/autor_datos.csv", "r", encoding="utf-8") as f:
    cursor.copy_expert("""
        COPY staging_authors (anho_nacimiento, nombre, fecha_creacion, total_obras)
        FROM STDIN WITH (FORMAT csv, HEADER true, DELIMITER ',')
    """, f)

# Cargar keys
with open("../data/autor_key.csv", "r", encoding="utf-8") as f:
    cursor.copy_expert("""
        COPY staging_keys (autor_key)
        FROM STDIN WITH (FORMAT csv, HEADER true, DELIMITER ',')
    """, f)

# Cargar nacionalidades
with open("../data/autor_nacionalidad.csv", "r", encoding="utf-8") as f:
    cursor.copy_expert("""
        COPY staging_nationalities (nacionalidad)
        FROM STDIN WITH (FORMAT csv, HEADER true, DELIMITER ',')
    """, f)

conn.commit()
print("CSVs cargados a staging")

CSVs cargados a staging


***CREACION DE SCHEMA DEFINITIVO PARA LA BASE DE DATOS***


In [33]:
schema_ddl = """
DROP TABLE IF EXISTS book_author CASCADE;
DROP TABLE IF EXISTS books CASCADE;
DROP TABLE IF EXISTS authors CASCADE;
DROP TABLE IF EXISTS categories CASCADE;

CREATE TABLE categories (
    id_category SERIAL PRIMARY KEY,
    category_name VARCHAR(80) NOT NULL UNIQUE
);

CREATE TABLE authors (
    id_author        SERIAL PRIMARY KEY,
    author_name      VARCHAR(100),
    birth_year       INTEGER,
    country          VARCHAR(100),
    external_api_id  VARCHAR(100),
    total_known_works INTEGER,
    api_source       VARCHAR(100)
);

CREATE TABLE books (
    id_book     SERIAL PRIMARY KEY,
    id_category INTEGER NOT NULL,
    title       VARCHAR(500),
    price       NUMERIC(12,2),
    rating      SMALLINT,
    CONSTRAINT fk_category FOREIGN KEY (id_category)
        REFERENCES categories(id_category)
);

CREATE TABLE book_author (
    id_book   INTEGER NOT NULL,
    id_author INTEGER NOT NULL,
    PRIMARY KEY (id_book, id_author),
    CONSTRAINT fk_book   FOREIGN KEY (id_book)   REFERENCES books(id_book),
    CONSTRAINT fk_author FOREIGN KEY (id_author) REFERENCES authors(id_author)
);

CREATE INDEX idx_books_category ON books(id_category);
CREATE INDEX idx_books_rating   ON books(rating);
CREATE INDEX idx_books_price    ON books(price);
CREATE INDEX idx_authors_name   ON authors(author_name);
CREATE INDEX idx_book_author_book ON book_author(id_book);
CREATE INDEX idx_book_author_auth ON book_author(id_author);
"""

cursor.execute(schema_ddl)
conn.commit()
print("Schema final creado")

Schema final creado


In [52]:
transform_load = """
-- 1. Categorías únicas
INSERT INTO categories (category_name)
SELECT DISTINCT TRIM(categoria)
FROM staging_books
WHERE categoria IS NOT NULL
ORDER BY TRIM(categoria);

-- 2. Autores únicos
INSERT INTO authors (author_name, birth_year, country, external_api_id, total_known_works, api_source)
SELECT DISTINCT
    TRIM(sa.nombre),
    NULLIF(SPLIT_PART(TRIM(sa.anho_nacimiento), '.', 1), '')::INTEGER,
    NULLIF(TRIM(sn.nacionalidad), ''),
    NULLIF(TRIM(sk.autor_key), ''),
    NULLIF(SPLIT_PART(TRIM(sa.total_obras), '.', 1), '')::INTEGER,
    'OpenLibrary + Wikipedia'
FROM staging_authors sa
JOIN staging_keys sk          ON sa.ctid = sk.ctid
JOIN staging_nationalities sn ON sa.ctid = sn.ctid
WHERE sk.autor_key IS NOT NULL;

-- 3. Libros (precio y calificacion ya limpios desde pandas)
INSERT INTO books (id_category, title, price, rating)
SELECT
    c.id_category,
    TRIM(sb.titulo),
    sb.precio::NUMERIC(12,2),
    sb.calificacion::SMALLINT
FROM staging_books sb
JOIN categories c ON TRIM(sb.categoria) = c.category_name;

-- 4. Relacion libro-autor
INSERT INTO book_author (id_book, id_author)
SELECT
    b.id_book,
    a.id_author
FROM staging_books sb
JOIN staging_keys sk ON sb.ctid = sk.ctid
JOIN books b         ON TRIM(sb.titulo) = b.title
JOIN authors a       ON TRIM(sk.autor_key) = a.external_api_id;
"""

cursor.execute(transform_load)
conn.commit()
print("Datos cargados correctamente")

Datos cargados correctamente


***VERIFICAMOS QUE LAS TABLAS FUERON CARGADAS CORRECTAMENTE***

In [53]:
cursor.execute("SELECT COUNT(*) FROM categories")
print("Categorías:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM authors")
print("Autores:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM books")
print("Libros:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM book_author")
print("Relaciones book_author:", cursor.fetchone()[0])

Categorías: 50
Autores: 553
Libros: 1000
Relaciones book_author: 596
